In [1]:
import random
import time
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


#  MAZE DEFINITIONS

MAZE1 = [
    ['S', 0, 1, 0, 0, 0, 1, 0, 0, 0],
    [ 1,  0, 1, 0, 1, 0, 1, 0, 1, 0],
    [ 0,  0, 0, 0, 1, 0, 0, 0, 1, 0],
    [ 0,  1, 1, 0, 1, 1, 1, 0, 1, 0],
    [ 0,  0, 0, 0, 0, 0, 1, 0, 0, 0],
    [ 1,  1, 1, 1, 1, 0, 1, 1, 1, 0],
    [ 0,  0, 0, 0, 1, 0, 0, 0, 0, 0],
    [ 0,  1, 1, 0, 1, 1, 1, 1, 1, 0],
    [ 0,  0, 1, 0, 0, 0, 0, 0, 1, 0],
    [ 1,  0, 0, 0, 1, 1, 1, 0, 0,'G'],
]

MAZE2 = [
    ['S', 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0],
    [ 1,  0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0],
    [ 0,  0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0],
    [ 0,  1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0],
    [ 0,  0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    [ 1,  1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0],
    [ 0,  0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
    [ 0,  1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0],
    [ 0,  0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0],
    [ 1,  0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0],
    [ 0,  0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0],
    [ 0,  1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0],
    [ 0,  0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0],
    [ 1,  1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0],
    [ 0,  0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [ 0,  1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0],
    [ 0,  0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0],
    [ 1,  0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0],
    [ 0,  0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0],
    [ 1,  0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0,'G'],
]

#List of directions
MOVES = ['U', 'D', 'L', 'R']


#  HELPER FUNCTIONS

#Finding the S and G coordinates inside the maze
def find_positions(maze):

    start = goal = None
    for i in range(len(maze)):
        for j in range(len(maze[0])):
            if maze[i][j] == 'S': start = (i, j)
            if maze[i][j] == 'G': goal  = (i, j)
    return start, goal

# Calculating how the coordinates U(Up), D(Down), L(Left), R(Right) will change according to the steps applied to the position.
def move(pos, action):

    x, y = pos
    if action == 'U': return (x - 1, y)
    if action == 'D': return (x + 1, y)
    if action == 'L': return (x, y - 1)
    if action == 'R': return (x, y + 1)

#A chromosome is placed in a maze and the result is returned. Invalid moves do not change position (wall, out of bounds).
def simulate(path, maze, start, goal):

    x, y = start #Start from the starting point
    wall_hits = out_of_bounds = actual_steps = 0  # The number of times you hit the wall, How many times have you left the boundry?, Number of steps completed
    visited    = {(x, y): 1} # How many times was each cell visited?
    path_order = [(x, y)] #sequential path (for arrow drawing)

    for action in path:
        nx, ny = move((x, y), action)

        # Have you escaped the boundaries of the maze?
        if nx < 0 or ny < 0 or nx >= len(maze) or ny >= len(maze[0]):
            out_of_bounds += 1
            continue

        # Checking how many times the wall was hit
        if maze[nx][ny] == 1:
            wall_hits += 1
            continue

        # Proceed if you haven't gone outside the area or hit a wall. Valid move.
        # The aim is to reduce the score in the fitness function. Returning to the same cell means entering a loop.
        x, y = nx, ny
        actual_steps += 1
        visited[(x, y)] = visited.get((x, y), 0) + 1
        path_order.append((x, y))

        # Stop if you've reached the goal
        if (x, y) == goal:
            break

    return (x, y), wall_hits, out_of_bounds, visited, actual_steps, path_order


#  FITNESS FUNCTION

def fitness(path, maze, start, goal, penalty=10):

    # simulate is called.
    pos, wall_hits, oob, visited, steps, _ = simulate(path, maze, start, goal)

    #The penalty for entering the loop.
    loop_penalty = sum(v - 1 for v in visited.values() if v > 1)

    #We reached the Goal, but points are calculated after deducting penalties.
    # We have 1000 points initially.
    if pos == goal:
        return (1000
                - steps        * 0.5  #every step
                - wall_hits    * penalty  #number of walls it hit
                - oob          * penalty  #going out of border
                - loop_penalty * (penalty * 0.5))  #every cycle

    #We didn't reach the Goal, but we'll get negative points depending on the distance.
    dist = abs(pos[0] - goal[0]) + abs(pos[1] - goal[1])
    return (-dist          * 2.0
            - wall_hits    * penalty
            - oob          * penalty
            - loop_penalty * (penalty * 0.5))


#  POPULATION INITIALIZATION

#Random individuals are selected. An initial population is created.
def create_population(pop_size, chrom_length):
    return [[random.choice(MOVES) for _ in range(chrom_length)]
            for _ in range(pop_size)]


#  SELECTION — Tournament Selection

#In each cycle, a number of individuals equal to tournament_size are randomly selected from the population.
#The individual with the best fitness value in the next tournament wins.
#This process is repeated until the population size reaches the selection pool.
#As tournament_size increases, the selection pressure becomes stronger. Diversity decreases, but convergence accelerates.

def tournament_select(population, fitnesses, tournament_size):

    n = len(population)
    selected = []
    for _ in range(n):
        #A random `tournament_size` candidate will be selected.
        idx    = random.sample(range(n), tournament_size)
        # The index of the individual with the highest fitness level among them is determined.
        winner = max(idx, key=lambda i: fitnesses[i])
        selected.append(population[winner][:])
    return selected

#  CROSSOVER — Single Point Crossover

#A random selection point is chosen.
#The left part of p1 and the right part of p2 are combined to create a new individual.

def crossover(p1, p2, rate):
    # crossover_rate (probability of crossover)
    if random.random() > rate:
        return p1[:]
    point = random.randint(1, len(p1) - 1)
    return p1[:point] + p2[point:]


#  MUTATION

# Each gene is independently modified in a random move with a probability of `rate`.
# A low rate provides stability, while a high rate provides diversity.

def mutate(individual, rate):
    return [random.choice(MOVES) if random.random() < rate else g
            for g in individual]


#  MAIN GA LOOP

def run_ga(maze, pop_size, chrom_length, generations,
           mutation_rate, crossover_rate, tournament_size,
           verbose=False):
    """
    Run the Genetic Algorithm on the given maze.

    Key design decisions:
    - Two fitness scales:
        * selection_penalty (escalating, max 160): drives selection pressure
        * FIXED_PENALTY = 10 (constant): used for logging and comparison
    - solved_path: once a path reaches the goal it is preserved via
      elitism and can never be lost in later generations.
    - Elitism: top individual + solved_path both carry over each generation.

    Returns:
        best_path, best_fitness, fitness_history,
        solved, solved_generation, elapsed,
        diversity_history, solved_count, plan_b_info
    """
    # Fixed penalty keeps fitness values stable and comparable across generations
    FIXED_PENALTY = 10
    start, goal   = find_positions(maze)
    population    = create_population(pop_size, chrom_length)

    best_path         = None
    best_fitness      = float('-inf')
    solved_path       = None   # best path that reaches the goal
    fitness_history   = [] # It preserves the best fitness values ​​of every generation
    diversity_history = [] # To report the results
    solved            = False
    solved_generation = None
    solved_count      = 0  # Counts how many generations it takes to reach the goal
    t0 = time.time()

    for gen in range(generations):

        # Escalating selection pressure (used only in tournament selection)
        # Fitness history and solved_individual always use FIXED_PENALTY
        sel_penalty = min(10 * (2 ** (gen // 30)), 160)

        # Fitness with escalating penalty (for selection pressure)
        fit_sel   = [fitness(ind, maze, start, goal, sel_penalty)   for ind in population]

        # Fitness with fixed penalty (for history and comparison)
        fit_fixed = [fitness(ind, maze, start, goal, FIXED_PENALTY) for ind in population]

        # Diversity; finds the number of unique individuals in the population.
        diversity_history.append(len(set(tuple(ind) for ind in population)))

        # Best individual in this generation (by selection penalty)
        best_idx     = max(range(pop_size), key=lambda i: fit_sel[i])
        gen_best_fit = fit_fixed[best_idx]
        fitness_history.append(gen_best_fit)

        # Update overall best (with fixed penalty)
        if gen_best_fit > best_fitness:
            best_fitness = gen_best_fit
            best_path    = population[best_idx][:]

        # Goal check
        # All individuals are checked; the first goal-reaching one is saved.
        # solved_individual is never lost once set.
        # Comparisons always use FIXED_PENALTY for consistency.
        for ind in population:
            pos, *_ = simulate(ind, maze, start, goal)
            if pos == goal:
                solved_count += 1
                if not solved:
                    solved            = True
                    solved_generation = gen
                    solved_path       = ind[:]
                    if verbose:
                        f = fitness(ind, maze, start, goal, FIXED_PENALTY)
                        print(f"  ✅ Goal first reached at generation {gen} "
                              f"(fitness={f:.1f})")
                # Keep the most efficient goal-reaching path
                if fitness(ind, maze, start, goal, FIXED_PENALTY) > \
                   fitness(solved_path, maze, start, goal, FIXED_PENALTY):
                    solved_path = ind[:]
                break  # one check per generation is sufficient

        if verbose and gen % 50 == 0:
            print(f"  Gen {gen:>4} | Fitness: {gen_best_fit:>10.1f} | "
                  f"Sel.Penalty: {sel_penalty:>4} | Diversity: {diversity_history[-1]:>4}")

        # Build next generation

        #1. Tournament selection (matting pool).
        mating_pool = tournament_select(population, fit_sel, tournament_size)

        #2. Elitism: the best individual is directly passed on to the next generation.
        new_pop = [population[best_idx][:]]   # elitism: best individual

        # Also preserve the goal-reaching individual (if any)
        if solved_path is not None:
            new_pop.append(solved_path[:])    # elitism: goal-reaching path

        #3) The remaining individuals: produced via crossover + mutation.
        while len(new_pop) < pop_size:
            p1 = random.choice(mating_pool)
            p2 = random.choice(mating_pool)
            new_pop.append(mutate(crossover(p1, p2, crossover_rate), mutation_rate))
        population = new_pop

    elapsed    = time.time() - t0
    final_path = solved_path if solved_path else best_path

    # Use solved_individual's fitness when goal is reached (consistency)
    if solved:
        final_fitness = fitness(solved_path, maze, start, goal, FIXED_PENALTY)
    else:
        final_fitness = best_fitness

    plan_b_info = None   # populated when goal is not reached

    # PLAN B: If the objective was not achieved
    if not solved:
        pos, *_ = simulate(final_path, maze, start, goal)
        dist = abs(pos[0] - goal[0]) + abs(pos[1] - goal[1])
        plan_b_info = {'pos': pos, 'dist': dist, 'fitness': final_fitness}
        if verbose:
            print(f"  ❌ Goal not reached.")
            print(f"     Final position : {pos}  |  Distance: {dist} cells")
            print(f"     Best fitness   : {final_fitness:.1f}")
    elif verbose:
        print(f"  ℹ️  Goal maintained for {solved_count} generation(s) "
              f"(first at generation {solved_generation}).")

    return (final_path, final_fitness, fitness_history,
            solved, solved_generation, elapsed,
            diversity_history, solved_count, plan_b_info)



#  PARAMETER TUNING  (ceteris paribus)

#  Logic: 3 values are tested for each parameter.
#  All other parameters remain fixed at baseline.
#  This isolates the effect of each individual parameter.

def run_parameter_tuning(maze, maze_name, baseline, param_ranges):

    configs = []
    for param, values in param_ranges.items():
        for val in values:
            cfg = {**baseline, param: val, '_param': param, '_value': val}
            configs.append(cfg)

    n_params = len(param_ranges)
    n_vals   = len(next(iter(param_ranges.values())))
    print(f"\n{'═'*90}")
    print(f"  PARAMETER TUNING — {maze_name}  "
          f"({n_params} parameters × {n_vals} values = {len(configs)} runs)")
    print(f"{'═'*90}")

    results       = []
    current_param = None

    for cfg in configs:
        param, value = cfg['_param'], cfg['_value']

        if param != current_param:
            current_param = param
            fixed_str = "  |  ".join(
                f"{k}={v}" for k, v in baseline.items() if k != param)
            print(f"\n\n  {'━'*85}")
            print(f"  VARYING: {param.upper()}")
            print(f"  Fixed  : {fixed_str}")
            print(f"  {'━'*85}")

        print(f"\n  ── {param} = {value} ──")
        print(f"    Running...", end='', flush=True)

        (path, fit, hist, solved, gen_no,
         elapsed, div, scount, plan_b) = run_ga(
            maze=maze,
            pop_size        = cfg['pop_size'],
            chrom_length    = cfg['chrom_length'],
            generations     = cfg['generations'],
            mutation_rate   = cfg['mutation_rate'],
            crossover_rate  = cfg['crossover_rate'],
            tournament_size = cfg['tournament_size'],
            verbose=False,
        )

        status  = '✅ SOLVED  ' if solved else '❌ FAILED  '
        gen_str = f"gen {gen_no:>4}" if solved else "gen    —  "
        print(f"  {status}  |  {gen_str}  |  {elapsed:.1f}s  |  fitness={fit:.1f}")
        if not solved and plan_b:
            print(f"         Plan B → pos={plan_b['pos']}  "
                  f"distance={plan_b['dist']} cells")

        results.append({
            'pop_size':        cfg['pop_size'],
            'chrom_length':    cfg['chrom_length'],
            'generations':     cfg['generations'],
            'mutation_rate':   cfg['mutation_rate'],
            'crossover_rate':  cfg['crossover_rate'],
            'tournament_size': cfg['tournament_size'],
            'param':   param,
            'value':   value,
            'solved':  solved,
            'gen':     gen_no,
            'fitness': fit,
            'time':    elapsed,
        })

    # Summary table
    print(f"\n\n{'═'*80}")
    print(f"  SUMMARY — {maze_name}")
    print(f"{'═'*80}")
    print(f"  {'Parameter':<18} {'Value':>7} {'Result':>12} "
          f"{'Gen':>6} {'Fitness':>10} {'Time(s)':>8}")
    print(f"  {'─'*75}")
    last = None
    for r in results:
        if r['param'] != last:
            if last: print(f"  {'─'*75}")
            last = r['param']
        status = '✅ SOLVED  ' if r['solved'] else '❌ FAILED  '
        gen    = str(r['gen']) if r['solved'] else '—'
        print(f"  {r['param']:<18} {str(r['value']):>7} {status:>12} "
              f"{gen:>6} {r['fitness']:>10.1f} {r['time']:>8.2f}")
    print(f"  {'─'*75}")

    return results



#  PLOTS

# PLOT 1: Convergence

#Plots and records convergence graphs for both mazes.
#Generation-based best fitness graph for both mazes.

def plot_convergence(hist1, hist2, filename="convergence.png"):
    plt.figure(figsize=(10, 5))
    plt.plot(hist1, label="Maze 1 (10×10)", color='steelblue', linewidth=1.5)
    plt.plot(hist2, label="Maze 2 (20×20)", color='tomato',    linewidth=1.5)
    plt.xlabel("Generation")
    plt.ylabel("Best Fitness")
    plt.title("Genetic Algorithm — Convergence Plot")
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
    plt.savefig(filename, dpi=150); plt.close()
    print(f"  → '{filename}' saved.")

# PLOT 2: Population Diversity
#Indicates the number of unique individuals in a population over a generation.

def plot_diversity(div1, div2, filename="diversity.png"):
    plt.figure(figsize=(10, 5))
    plt.plot(div1, label="Maze 1 (10×10)", color='steelblue', linewidth=1.5)
    plt.plot(div2, label="Maze 2 (20×20)", color='tomato',    linewidth=1.5)
    plt.xlabel("Generation")
    plt.ylabel("Unique Individuals")
    plt.title("Genetic Algorithm — Population Diversity")
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
    plt.savefig(filename, dpi=150); plt.close()
    print(f"  → '{filename}' saved.")

# PLOT 3: Maze Solution Path Visualization

def plot_maze_solution(maze, path, maze_name, filename):
    rows, cols  = len(maze), len(maze[0])
    start, goal = find_positions(maze)

    # Run the GA simulation; find which cells it used.
    pos, _, _, visited, _, path_order = simulate(path, maze, start, goal)

    # Remove duplicate cells but maintain the order
    seen, unique_path = set(), []
    for cell in path_order:
        if cell not in seen:
            seen.add(cell); unique_path.append(cell)

    fig, ax = plt.subplots(figsize=(cols * 0.65 + 1.5, rows * 0.65 + 1.5))
    COLOR = {'wall': '#2c2c2c', 'start': '#3a86ff',
             'goal': '#06d6a0', 'path':  '#ffb703', 'free': '#f8f9fa'}

    for r in range(rows):
        for c in range(cols):
            cell = maze[r][c]
            if   cell == 1:        color = COLOR['wall']
            elif (r,c) == start:   color = COLOR['start']
            elif (r,c) == goal:    color = COLOR['goal']
            elif (r,c) in visited: color = COLOR['path']
            else:                  color = COLOR['free']
            ax.add_patch(mpatches.FancyBboxPatch(
                (c, rows-r-1), 1, 1, boxstyle="square,pad=0",
                facecolor=color, edgecolor='#adb5bd', linewidth=0.5))
            if cell in ('S', 'G'):
                ax.text(c+0.5, rows-r-0.5, str(cell),
                        ha='center', va='center',
                        fontsize=9, fontweight='bold', color='white')

    # Drawing an arrow on the path
    # A small arrow is drawn between each consecutive pair of cells.
    # These arrows clearly show the direction of travel.

    ap = dict(arrowstyle='->', color='#d62828', lw=1.2, mutation_scale=8)
    for i in range(len(unique_path) - 1):
        r1, c1 = unique_path[i]; r2, c2 = unique_path[i+1]
        ax.annotate("", xy=(c2+0.5, rows-r2-0.5),
                    xytext=(c1+0.5, rows-r1-0.5), arrowprops=ap)

    ax.set_xlim(0, cols); ax.set_ylim(0, rows)
    ax.set_aspect('equal'); ax.axis('off')
    ax.legend(handles=[
        mpatches.Patch(facecolor=COLOR['start'], label='Start (S)'),
        mpatches.Patch(facecolor=COLOR['goal'],  label='Goal (G)'),
        mpatches.Patch(facecolor=COLOR['path'],  label='GA Path'),
        mpatches.Patch(facecolor=COLOR['wall'],  label='Wall'),
        mpatches.Patch(facecolor=COLOR['free'],  edgecolor='#adb5bd', label='Free Cell'),
    ], loc='upper right', bbox_to_anchor=(1.22, 1), fontsize=8)
    ax.set_title(f"{maze_name} — {'[SOLVED]' if pos == goal else '[FAILED]'}",
                 fontsize=11, pad=10)
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight'); plt.close()
    print(f"  → '{filename}' saved.")


#  CONFIGURATION

#  Maze 1 Parameters
# Baseline: moderate values that reliably solve the maze
BASELINE_1 = {
    'pop_size': 80, 'chrom_length': 40, 'generations': 300,
    'mutation_rate': 0.05, 'crossover_rate': 0.8, 'tournament_size': 4,
}

# 3 values per parameter
PARAM_RANGES_1 = {
    'pop_size':        [30,   80,  150],
    'chrom_length':    [20,   40,   60],
    'generations':     [100, 300,  500],
    'mutation_rate':   [0.01, 0.05, 0.10],
    'crossover_rate':  [0.5,  0.8,  0.95],
    'tournament_size': [2,    4,    7],
}

# Maze 2 Parameters
# Baseline: moderate level
BASELINE_2 = {
    'pop_size': 200, 'chrom_length': 120, 'generations': 500,
    'mutation_rate': 0.07, 'crossover_rate': 0.85, 'tournament_size': 5,
}

# 3 values per parameter
PARAM_RANGES_2 = {
    'pop_size':        [150,   200,  400],
    'chrom_length':    [70,   120,  180],
    'generations':     [200,  500,  800],
    'mutation_rate':   [0.01, 0.07, 0.15],
    'crossover_rate':  [0.5,  0.85, 0.95],
    'tournament_size': [2,    5,    8],
}


#  ENTRY POINT

if __name__ == "__main__":

    # 1 — Parameter tuning
    results1 = run_parameter_tuning(MAZE1, "MAZE 1 — 10×10", BASELINE_1, PARAM_RANGES_1)
    results2 = run_parameter_tuning(MAZE2, "MAZE 2 — 20×20", BASELINE_2, PARAM_RANGES_2)

    # 2 — Pick best config from tuning results
    #     Priority: solved configs (highest fitness), then unsolved (highest fitness)
    def best_config(results):
        solved = [r for r in results if r['solved']]
        pool   = solved if solved else results
        return max(pool, key=lambda r: r['fitness'])

    cfg1 = best_config(results1)
    cfg2 = best_config(results2)

    # 3 — Final run with best config
    final = {}
    for maze, cfg, label, fname in [
        (MAZE1, cfg1, "MAZE 1 — 10×10", "Maze 1 (10×10)"),
        (MAZE2, cfg2, "MAZE 2 — 20×20", "Maze 2 (20×20)"),
    ]:
        print(f"\n{'═'*60}")
        print(f"  FINAL RUN — {label}")
        print(f"  Best config → pop={cfg['pop_size']}, chrom={cfg['chrom_length']}, "
              f"gen={cfg['generations']}, mut={cfg['mutation_rate']}, "
              f"cr={cfg['crossover_rate']}, tour={cfg['tournament_size']}")
        print(f"{'═'*60}")

        (path, fit, hist, solved, gen_no,
         elapsed, div, scount, _) = run_ga(
            maze=maze, verbose=True,
            pop_size        = cfg['pop_size'],
            chrom_length    = cfg['chrom_length'],
            generations     = cfg['generations'],
            mutation_rate   = cfg['mutation_rate'],
            crossover_rate  = cfg['crossover_rate'],
            tournament_size = cfg['tournament_size'],
        )
        print(f"  Time: {elapsed:.2f}s | Solved: {solved} | "
              f"First gen: {gen_no} | Stable count: {scount}")
        final[label] = dict(path=path, fit=fit, hist=hist, solved=solved,
                            gen=gen_no, time=elapsed, div=div, scount=scount,
                            name=fname)

    # 4 — Summary comparison
    m1, m2 = final["MAZE 1 — 10×10"], final["MAZE 2 — 20×20"]
    print(f"\n{'═'*62}")
    print(f"  SUMMARY — Maze Comparison")
    print(f"{'═'*62}")
    print(f"  {'Metric':<30} {'Maze 1 (10×10)':>14} {'Maze 2 (20×20)':>14}")
    print(f"  {'─'*58}")
    print(f"  {'Solved?':<30} {str(m1['solved']):>14} {str(m2['solved']):>14}")
    print(f"  {'First solution generation':<30} {str(m1['gen']):>14} {str(m2['gen']):>14}")
    print(f"  {'Stable solution count':<30} {str(m1['scount']):>14} {str(m2['scount']):>14}")
    print(f"  {'Best fitness':<30} {m1['fit']:>14.1f} {m2['fit']:>14.1f}")
    print(f"  {'Runtime (s)':<30} {m1['time']:>14.2f} {m2['time']:>14.2f}")
    print(f"  {'─'*58}")

    # 5 — Plots
    print("\n  Generating plots...")
    plot_convergence(m1['hist'], m2['hist'])
    plot_diversity(m1['div'], m2['div'])
    plot_maze_solution(MAZE1, m1['path'], m1['name'], "maze1_solution.png")
    plot_maze_solution(MAZE2, m2['path'], m2['name'], "maze2_solution.png")
    print("\n  Done.")


══════════════════════════════════════════════════════════════════════════════════════════
  PARAMETER TUNING — MAZE 1 — 10×10  (6 parameters × 3 values = 18 runs)
══════════════════════════════════════════════════════════════════════════════════════════


  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  VARYING: POP_SIZE
  Fixed  : chrom_length=40  |  generations=300  |  mutation_rate=0.05  |  crossover_rate=0.8  |  tournament_size=4
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  ── pop_size = 30 ──
    Running...  ✅ SOLVED    |  gen  232  |  1.4s  |  fitness=927.0

  ── pop_size = 80 ──
    Running...  ✅ SOLVED    |  gen    6  |  3.4s  |  fitness=957.0

  ── pop_size = 150 ──
    Running...  ✅ SOLVED    |  gen    5  |  3.6s  |  fitness=974.0


  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  VARYING: CHROM_LENGTH
  Fixed  : pop_size=80  |  generations=300  | 